# ✂️ 이미지 분할 (Image Segmentation)

MediaPipe Image Segmenter를 사용하여 이미지에서 **전경/배경을 분할**합니다.

```bash
pip install mediapipe opencv-python numpy
```

| 항목 | 내용 |
|------|------|
| **태스크** | `ImageSegmenter` |
| **모델** | `models/selfie_segmenter.tflite` (Selfie Segmentation) |
| **출력** | 분할 마스크 (각 픽셀의 클래스 확률) |
| **활용** | 배경 제거(블러), 셀피 보정, 가상 배경 |

---
## 1. Selfie Segmentation 이해

Selfie Segmentation 모델은 이미지에서 **사람(전경)**과 **배경**을 분리합니다.

**출력 형식:**
- `result.confidence_masks`: 각 클래스의 신뢰도 마스크 (0.0~1.0)
- `result.category_mask`: 각 픽셀의 카테고리 인덱스
- 마스크 크기: 원본 이미지와 동일

**활용 예:**
- Zoom/Teams 가상 배경
- 인물 사진 배경 블러 (아이폰 인물 모드)
- 배경 투명화 (PNG)
- 그린스크린 대체

In [ ]:
# 이미지 분할 — 정적 이미지 처리
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

# 모델 로드
model_path = 'models/selfie_segmenter.tflite'
base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
options = vision.ImageSegmenterOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    output_category_mask=True,
    output_confidence_masks=True,
)

# 테스트 이미지 생성
test_img = np.zeros((480, 640, 3), dtype=np.uint8)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=test_img)

# 추론 실행
with vision.ImageSegmenter.create_from_options(options) as segmenter:
    result = segmenter.segment(mp_image)
    print(f'신뢰도 마스크 수: {len(result.confidence_masks)}')
    if result.confidence_masks:
        for i, mask in enumerate(result.confidence_masks):
            mask_np = mask.numpy_view()
            print(f'  마스크[{i}]: shape={mask_np.shape}, '
                  f'min={mask_np.min():.3f}, max={mask_np.max():.3f}')
    if result.category_mask:
        cat_mask = result.category_mask.numpy_view()
        print(f'카테고리 마스크: shape={cat_mask.shape}, '
              f'고유값={np.unique(cat_mask)}')
        print('  0=배경, 1=전경(사람)')

---
## 2. 배경 블러 처리 (인물 모드)

분할 마스크를 사용하여 배경만 블러 처리하고 사람은 선명하게 유지합니다.
스마트폰의 인물 모드와 같은 효과를 만듭니다.

In [ ]:
# 배경 블러 함수 (인물 모드 효과)
import cv2
import numpy as np

def apply_background_blur(image, confidence_mask, blur_strength=25):
    """배경을 블러 처리하고 전경은 선명하게 유지합니다."""
    mask = (confidence_mask.numpy_view() * 255).astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)
    blurred = cv2.GaussianBlur(image, (blur_strength, blur_strength), 0)
    mask_3ch = cv2.merge([binary_mask, binary_mask, binary_mask])
    result = np.where(mask_3ch == 255, image, blurred)
    return result

# 테스트
test_img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

class FakeMask:
    def numpy_view(self):
        m = np.zeros((480, 640), dtype=np.float32)
        cv2.circle(m, (320, 240), 150, 1.0, -1)
        return m

result_img = apply_background_blur(test_img, FakeMask(), blur_strength=35)
print(f'배경 블러 처리 완료: {result_img.shape}')

---
## 3. 배경 교체 (가상 배경)

분할 마스크를 사용하여 배경을 다른 이미지나 색상으로 교체합니다.

In [ ]:
# 배경 교체 함수
import cv2
import numpy as np

def replace_background(image, confidence_mask, bg_color=(0, 0, 255)):
    """배경을 지정한 색상으로 교체합니다."""
    mask = (confidence_mask.numpy_view() * 255).astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)
    bg = np.full_like(image, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([binary_mask, binary_mask, binary_mask])
    result = np.where(mask_3ch == 255, image, bg)
    return result

def replace_background_image(image, confidence_mask, bg_image):
    """배경을 다른 이미지로 교체합니다."""
    h, w = image.shape[:2]
    bg_resized = cv2.resize(bg_image, (w, h))
    mask = (confidence_mask.numpy_view() * 255).astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)
    mask_blur = cv2.GaussianBlur(binary_mask, (5, 5), 0)
    mask_3ch = cv2.merge([mask_blur, mask_blur, mask_blur]).astype(np.float32) / 255
    result = (image.astype(np.float32) * mask_3ch + 
              bg_resized.astype(np.float32) * (1 - mask_3ch)).astype(np.uint8)
    return result

# 테스트
test_img = np.random.randint(100, 200, (480, 640, 3), dtype=np.uint8)
green_bg = replace_background(test_img, FakeMask(), bg_color=(0, 255, 0))
print(f'배경 교체(초록) 완료: {green_bg.shape}')
bg_img = np.random.randint(0, 256, (600, 800, 3), dtype=np.uint8)
replaced = replace_background_image(test_img, FakeMask(), bg_img)
print(f'배경 교체(이미지) 완료: {replaced.shape}')

---
## 4. 배경 투명화 (PNG)

분할 마스크를 사용하여 배경을 투명하게 만들고 PNG로 저장합니다.

In [ ]:
# 배경 투명화 함수
import cv2
import numpy as np

def make_transparent_background(image, confidence_mask):
    """배경을 투명하게 만들어 BGRA 이미지를 반환합니다."""
    mask = (confidence_mask.numpy_view() * 255).astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)
    alpha = cv2.GaussianBlur(binary_mask, (3, 3), 0)
    bgra = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)
    bgra[:, :, 3] = alpha
    return bgra

# 테스트
test_img = np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8)
transparent = make_transparent_background(test_img, FakeMask())
print(f'투명 배경 이미지: shape={transparent.shape} (4 채널)')
print(f'알파 채널 범위: {transparent[:, :, 3].min()} ~ {transparent[:, :, 3].max()}')
# cv2.imwrite('output_transparent.png', transparent)
print('실제 이미지로 테스트하세요.')

---
## 5. 웹캠 실시간 배경 처리

웹캠에서 실시간으로 배경을 블러 처리하거나 교체합니다.

In [ ]:
# 웹캠 실시간 배경 블러 (참고용)
# import mediapipe as mp
# from mediapipe.tasks.python import vision
# import cv2
# import time

# model_path = 'models/selfie_segmenter.tflite'
# base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
# options = vision.ImageSegmenterOptions(
#     base_options=base_options,
#     running_mode=vision.RunningMode.LIVE_STREAM,
#     output_confidence_masks=True,
#     result_callback=lambda r, img, ts: None
# )

# cap = cv2.VideoCapture(0)
# with vision.ImageSegmenter.create_from_options(options) as seg:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret: break
#         mp_image = mp.Image(mp.ImageFormat.SRGB, frame)
#         ts = int(time.time() * 1000)
#         seg.segment_async(mp_image, ts)
#         if seg.latest_results and seg.latest_results.confidence_masks:
#             frame = apply_background_blur(frame, seg.latest_results.confidence_masks[0])
#         cv2.imshow('BG Blur', frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'): break
# cap.release()
# cv2.destroyAllWindows()
print('웹캠 코드는 주석 처리됨')

---
## 6. 분할 마스크 시각화

분할 마스크를 컬러 오버레이로 시각화합니다.

In [ ]:
# 마스크 오버레이 시각화
import cv2
import numpy as np

def visualize_mask_overlay(image, confidence_mask, fg_color=(0, 255, 0)):
    """분할 마스크를 반투명 컬러 오버레이로 표시합니다."""
    mask = confidence_mask.numpy_view()
    overlay = image.copy()
    overlay_color = np.zeros_like(image)
    overlay_color[:] = fg_color
    mask_3ch = np.stack([mask, mask, mask], axis=2)
    overlay = np.where(mask_3ch > 0.5, 
                       cv2.addWeighted(image, 0.5, overlay_color, 0.5, 0),
                       image)
    return overlay.astype(np.uint8)

# 테스트
test_img = np.random.randint(80, 180, (480, 640, 3), dtype=np.uint8)
overlay = visualize_mask_overlay(test_img, FakeMask())
print(f'마스크 오버레이 완료: {overlay.shape}')
print('전경(사람) 영역이 초록색 반투명으로 표시됩니다.')

---
## 🎯 정리

| 기능 | 코드 | 설명 |
|------|------|------|
| 분할 실행 | `segmenter.segment(mp_image)` | 이미지 분할 |
| 신뢰도 마스크 | `result.confidence_masks[0]` | 0.0~1.0 확률 마스크 |
| 카테고리 마스크 | `result.category_mask` | 픽셀별 클래스 인덱스 |
| 마스크 → NumPy | `mask.numpy_view()` | NumPy 배열로 변환 |
| 배경 블러 | `cv2.GaussianBlur` + 마스크 | 인물 모드 효과 |
| 배경 교체 | `np.where(mask, fg, bg)` | 가상 배경 |
| 투명 배경 | BGRA 알파 채널 | PNG 투명 이미지 |

**활용 분야:**
| 분야 | 응용 |
|------|------|
| 화상 회의 | 가상 배경 (Zoom, Teams) |
| 사진 보정 | 인물 모드 (배경 블러) |
| 콘텐츠 제작 | 배경 제거 (크로마키 대체) |
| e-커머스 | 상품 배경 제거 (투명 PNG) |

---
## 🎯 연습 문제

1. **가상 배경 갤러리**: 3개의 배경 이미지(사무실, 해변, 우주)를 준비하고 키보드 1/2/3으로 배경을 실시간 전환하는 코드를 작성하세요.

2. **가장자리 개선**: 마스크 가장자리의 계단 현상을 줄이기 위해 모폴로지 연산(열기/닫기)을 적용하여 가장자리를 부드럽게 만드세요. (힌트: `cv2.morphologyEx`)

3. **분할 + 객체 탐지 조합**: Selfie Segmenter로 사람 영역을 추출하고, 그 영역 내에서만 Object Detector를 실행하여 성능을 개선하는 파이프라인을 설계하세요.

4. **모자이크 처리**: 사람 얼굴 영역에만 모자이크를 적용하는 함수를 작성하세요. (힌트: Face Detector로 얼굴 위치를 찾고 해당 영역만 픽셀화)